# Hybrid Link Protection (HLP & TA-HLP) — Cloud Evaluation

**Author:** Md. Iqbal Haider Khan ([@ihkokil](https://github.com/ihkokil) / ihkokil@gmail.com)  
**Base Paper:** Haijun Geng *et al.*, *"A Hybrid Link Protection Scheme for Ensuring Network Service Availability in Link-State Routing Networks"*, IEEE JCN 2020.  

### Overview
This notebook provides a complete, self-contained environment to reproduce and evaluate:
1. **Baseline Algorithms:** OSPF, LFC (Loop-Free Criterion), TBFH, MNP, DMPA-e, MNP-e (Base Paper Tables 3 & 4).
2. **HLP (Original):** Stage 1 (MNP-e) + Stage 2 (Topological Availability-based Backup Path Protection).
3. **TA-HLP (Traffic-Adaptive):** Traffic-weighted BPP prioritization with gravity demand matrix.
4. **Full Statistical Evaluation:** Multi-replicate Monte-Carlo failure simulations across Rocketfuel ISP topologies with paired t-tests, Bonferroni correction, and effect size calculations.

---  
> 💡 **Kaggle Hardware Tip:** Set Accelerator to **None (CPU)**. Kaggle provides **4 CPU cores and ~30GB RAM**, which is optimal for graph algorithms and Dijkstra simulations.

In [ ]:
# Install / verify required dependencies
!pip install -q networkx numpy pandas matplotlib seaborn scipy
import os
import sys
import glob
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print("Python environment ready!")

In [ ]:
# Unpack complete codebase & topologies if not already in directory
import base64
import zlib
import json

packed_archive = '''...'''

if not os.path.exists('src'):
    print('Unpacking self-contained codebase and sample topologies...')
    raw_json = zlib.decompress(base64.b64decode(packed_archive)).decode('utf-8')
    files = json.loads(raw_json)
    for filepath, content in files.items():
        os.makedirs(os.path.dirname(filepath) or '.', exist_ok=True)
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(content)
    print(f'Successfully extracted {len(files)} project modules and topologies!')
else:
    print('Project files already present.')

os.makedirs('results', exist_ok=True)
os.makedirs('plots', exist_ok=True)
os.makedirs('data/rocketfuel', exist_ok=True)


## 1. Quick Interactive Demo
Run a quick test on the Abilene topology with a budget of 8 backup links.

In [ ]:
!python main.py --topology data/gml/Abilene.gml --budget 8 --trials 200

## 2. Base Paper Reproduction: Tables 3 & 4
Reproduce compute time and network availability across all 6 methods from Geng *et al.* (2020):
- **OSPF** (single SPT baseline)
- **LFC** (Loop-Free Criterion)
- **TBFH** (Two-Branch Fast Handoff)
- **MNP** (Multiple Next-hop Protection)
- **DMPA-e** (Disjoint Multi-Path Algorithm)
- **MNP-e** (Extended MNP)

In [ ]:
# Run reproduction across Abilene + 4 Rocketfuel topologies
!python -m experiments.run_table_reproduction --repeat 5 --trials 200
!python -m plotting.plot_table_reproduction

In [ ]:
# Display Table 3 (Compute Time) and Table 4 (Availability) CSVs
if os.path.exists('results/table3_compute_time.csv'):
    print('=== Table 3: Compute Time per Node (µs) ===')
    display(pd.read_csv('results/table3_compute_time.csv'))

if os.path.exists('results/table4_availability.csv'):
    print('\n=== Table 4: Network Service Availability (%) ===')
    display(pd.read_csv('results/table4_availability.csv'))

# Display Bar Charts
if os.path.exists('plots/table3_compute_time_bars.png'):
    display(Image('plots/table3_compute_time_bars.png'))
if os.path.exists('plots/table4_availability_bars.png'):
    display(Image('plots/table4_availability_bars.png'))


## 3. Overhead Budget Sweep & Availability Curves
Sweeps the backup-link budget $B \in \{2, 4, 6, 8, 10, 12, 14, 16\}$ to evaluate Traffic Availability vs Overhead.

In [ ]:
# Run budget sweep on Abilene, Geant2012, and Sprint
!python main.py --sweep --topologies Abilene.gml Geant2012.gml Sprint.gml --budgets 2 4 6 8 10 12 14 16 --trials 200 --replicates 5 --plot

In [ ]:
for p in sorted(glob.glob('plots/*traffic_avail*.png')):
    print(f'Displaying: {p}')
    display(Image(p))


## 4. Full Statistical Evaluation & Ablation Suite
Runs multi-replicate Monte-Carlo simulations on synthetic size-matched Rocketfuel topologies:
- **Exodus** (79 nodes, 147 edges)
- **Telstra** (108 nodes, 153 edges)
- **Tiscali** (161 nodes, 328 edges)
- **Sprint315** (315 nodes, 972 edges)

Includes paired one-sided t-tests, Bonferroni correction, Cohen's d effect sizes, and ablation studies.

In [ ]:
# Full statistical evaluation (Exodus & Telstra by default for quick execution, or omit --topologies for all 4)
!python -m experiments.run_statistical_experiments \
    --topologies Exodus Telstra \
    --budgets 2 4 6 8 10 12 14 16 \
    --trials 100 \
    --replicates 5 \
    --seed 42

In [ ]:
# Inspect Statistical Comparisons
stat_files = glob.glob('results*/*statistical_comparisons*.csv')
if stat_files:
    df_stat = pd.read_csv(stat_files[0])
    print('=== Statistical Comparisons (TA-HLP vs HLP) ===')
    display(df_stat)

underperform_files = glob.glob('results*/*never_underperforms*.csv')
if underperform_files:
    df_under = pd.read_csv(underperform_files[0])
    print('\n=== Underperformance Verification Check ===')
    display(df_under)


## 5. Export Results & Download Archive
Create a downloadable `.zip` archive containing all generated CSV tables and PNG figures.

In [ ]:
import shutil

# Create downloadable zip archive in Kaggle working directory
archive_name = '/kaggle/working/hlp_results_and_plots'
if not os.path.exists('/kaggle/working'):
    archive_name = 'hlp_results_and_plots'

shutil.make_archive(archive_name, 'zip', root_dir='.', base_dir='results')
print(f'Archive ready for download: {archive_name}.zip')
